# 一、前言

先进行理论学习，再用代码进行验证对应的结论，最后进行小结测试

# 二、理论核心

问题设定（§2.1）：数据由隐变量 z 生成：z ~ p(z)，x ~ p(x|z)。我们要学生成模型参数 θ，但边缘似然 p(x) = ∫p(x|z)p(z)dz 是高维积分，算不出来——这就是"intractable"。

解法（§2.2）：引入近似后验 q(z|x)（编码器），对每个数据点：<br>
log p(x) = KL(q(z|x) || p(z|x)) + ELBO(x)

KL ≥ 0，所以 log p(x) ≥ ELBO(x)。最大化 ELBO 就逼近了最大似然，ELBO 又可拆成两项：<br>
ELBO(x) = E_{q(z|x)}[log p(x|z)]  −  KL(q(z|x) || p(z))<br>
 ---------└── 重构项（解码器）─┘----└─ 正则项（让 q 靠近先验）─┘<br>
这是全文最重要的公式（ Kingma §2.3 等式 7）

难点（§2.2 末尾）：对 q(z|x) 采样求梯度，噪声 z 挡住了梯度路径——naïve Monte Carlo 梯度估计器方差巨大，不可用。

重参数化技巧（§2.4，全文核心）：高斯情形下把采样改写为确定性变换：<br>
z = μ + σ ⊙ ε,   ε ~ N(0, I)<br>
随机性全被装进 ε，μ 和 σ 变成可微的确定性路径，梯度正常回传。论文原话："It is often possible to express the random variable as a deterministic variable... a differentiable transformation of an auxiliary noise variable."（§2.4 三个可重参数化的分布家族：逆 CDF、位置-尺度族、复合变换）

# 三、代码验证

## 1.验证高斯 KL 解析式

In [28]:
import torch
def kl_gauss(mu1, logvar1, mu2, logvar2):
    """KL(N(mu1, exp(logvar1)) || N(mu2, exp(logvar2)))，闭式解"""
    mu1, logvar1, mu2, logvar2 = map(torch.as_tensor, (mu1, logvar1, mu2, logvar2))
    # torch.as_tensor 就是用来兼容你后面 kl_gauss(..., 0.0, 0.0) 直接传浮点数的，自动转成张量，.exp()就不会因为炸掉而报错
    var1, var2 = logvar1.exp(), logvar2.exp()
    return 0.5 * (logvar2 - logvar1 + (var1 + (mu1 - mu2) ** 2) / var2 - 1)
# 先验 p(z) = N(0, 1)，后验 q(z|x) = N(mu, sigma^2)，验证 KL >= 0 且仅在 q==p 时为 0
print(kl_gauss(torch.tensor(1.2), torch.tensor(0.5),
               torch.tensor(0.0), torch.tensor(0.0)))   # > 0

# q 和 p 完全相同，KL = 0
print(kl_gauss(torch.tensor(0.0), torch.tensor(0.0),
               torch.tensor(0.0), torch.tensor(0.0)))   # = 0

tensor(0.7944)
tensor(0.)


要点：VAE 里先验固定为 N(0,1)，所以上式退化为 KL = 0.5 * Σ(μ² + σ² − log σ² − 1)。为什么这样：两项 KL 都是正项，q 偏离先验越远惩罚越大。

var1, var2 = logvar1.exp(), logvar2.exp()<br>
logvar = ln (方差)<br>
`exp()` 就是 e 的 x 次方。<br>
因为 var = e^(logvar)<br>
也就是：方差 = e^( ln (方差) )<br>
作用：**把 log‑方差还原回正常方差**

为什么 VAE 编码器输出 logvar，不直接输出方差？<br>
方差有一个硬性要求：**必须大于 0** <br>
神经网络的输出可以是任意负数。<br>
1. 如果网络直接输出方差：一旦输出负数 → 方差为负，程序报错，高斯分布无意义。<br>
2. 如果网络输出 logvar：<br>
网络随便输出什么数字，正数负数都行。<br>
再执行 `.exp()`，结果永远 > 0。<br>
方差就一定合法。<br>
总结：输出 logvar 是**保证方差恒正的安全技巧**。

VAE 最常使用的简化版（必背）<br>
VAE 规定隐变量先验 p (z) 永远是 N (0,1)<br>
也就是 mu2=0，logvar2=0，var2=1<br>
代入上面公式化简：<br>
KL (q||p) = 0.5 * ( -logvar1 + mu1**2 + var1 - 1 )<br>
以后写 VAE 损失直接抄这个式子。

三个参数区分清单<br>
1. sigma → 标准差<br>
2. var → 方差 = sigma * sigma<br>
3. logvar → ln (var)，编码器输出

## 2. 重参数化 vs 直接采样（演示"为什么必须重参数化"）

In [29]:
mu = torch.tensor(0.0, requires_grad = True)    # 正态分布均值，网络输出参数，**需要被优化**，打开 `requires_grad=True`
sigma = torch.tensor(1.0, requires_grad = True)   # 正态分布标准差，网络输出参数，需要被优化
dist = torch.distributions.Normal(mu, sigma)    # 创建正态分布：z ~ N(mu,sigma^2)

# ----------方式1：sample() 普通采样，梯度断了----------
z = dist.sample()  # 直接从正态分布随机抽出一个数 z
loss = z ** 2
try:
    loss.backward()
    print('mu.grad =', mu.grad)   # None，因为 sample 路径不可微
except Exception as e:
    print('sample() 报错:', type(e).__name__)
    
# ----------方式2：rsample() 重参数采样，梯度流通----------
mu.grad = None   # 清空上一轮残留梯度
z = dist.rsample()
loss = z ** 2
loss.backward()
print('rsample 后 mu.grad =', mu.grad)  # 有值：2*(mu + sigma*eps)

sample() 报错: RuntimeError
rsample 后 mu.grad = tensor(2.4643)


要点：rsample() 内部就是 mu + sigma * eps——论文 §2.4 的高斯例子原样复刻。不重参数化会怎样：梯度为 None/报错，训练完全无法进行。这就是 VAE 论文能成立的技术基石。多跑几次：每次 mu.grad 都不同（因为 ε 是随机噪声），但均值 ≈ 2μ=0——这就是"无偏估计器"的含义。

`sample()` 为什么梯度传不回去？<br>
z = dist.sample()<br>
`sample()` 直接从正态分布随机抽出一个数 z。<br>
数学上：<br>
$z \sim \mathcal{N}(\mu,\sigma^2)$ <br>
z 是**随机采样结果**，不是一个用 $\mu$、$\sigma$ 写成的显式计算公式。<br>
计算图看不到 z 和 $\mu,\sigma$ 的函数依赖关系，**梯度链路断掉**。<br>
所以执行 `loss.backward()` 之后：
`mu.grad` = `None`，得不到梯度，神经网络没法更新 $\mu$ 和 $\sigma$。<br>
> 总结：<br>
> 直接抓一个随机数 z，PyTorch 不知道这个 z 是怎么由 mu、sigma 算出来的，不知道该怎么求导。

重参数技巧（Reparameterization Trick）解决办法<br>
把采样拆成两步：<br>
1. 先从**标准正态**采噪声：$\varepsilon \sim \mathcal{N}(0, 1)$，这个 $\varepsilon$ 和 $\mu,\sigma$ **毫无关系**，不需要梯度<br>
2. 再手动变换：<br>
$\boldsymbol{z=\mu+\sigma\cdot\varepsilon}$ <br>
现在 z 就变成了 $\mu$ 和 $\sigma$ 的**显式函数**！<br>
现在 z 和两个参数之间的求导关系完全可见，梯度就可以流回去了。<br>
`rsample()` 就是帮你自动执行上面两步的函数。<br>
> rsample = re‑parameterization sample，重参数采样

求导验算（对应代码 loss = z²）<br>
$loss = z^2 = (\mu + \sigma\,\varepsilon)^2$对 $\mu$ 求导：<br>
$\frac{\partial loss}{\partial \mu}=2(\mu+\sigma\,\varepsilon)$ <br>
反向传播之后 `mu.grad` 就会得到上面这个数值。

## 3.验证"为什么 ELBO 是下界"（用离散近似积分直接算）

In [31]:
x = torch.tensor(1.0)                # 一个观测数据点
mu_q, logvar_q = torch.tensor(1.0), torch.tensor(0.0)   # 编码器给出后验: q(z|x) = N(1, 1)

q = torch.distributions.Normal(mu_q, logvar_q.exp().sqrt())    # .sqrt()：方差开根号得到标准差，Normal 第二个参数接收标准差。
zs = q.rsample((20000,))              # 从 q 采 2 万个 z（重参数化）,rsample = 重参数采样，采样结果可回传梯度
log_p_xz = torch.distributions.Normal(zs, 1.0).log_prob(x) # log N(x; z, 1)
elbo_recon = log_p_xz.mean()

elbo_kl = kl_gauss(mu_q, logvar_q, 0.0, 0.0)
elbo = elbo_recon - elbo_kl

# 数值积分求真实边缘似然 log p (x)
z_grid = torch.linspace(-10, 10, 100000)  # 在区间 [-10, 10] 取10万离散网格点
p_xz = torch.distributions.Normal(z_grid, 1.0).log_prob(x).exp()   # p(x|z)
p_z = torch.distributions.Normal(0.0, 1.0).log_prob(z_grid).exp()   # 先验 p(z)
trapezoid = getattr(torch, 'trapezoid', torch.trapz)
log_p_x = trapezoid(p_xz * p_z, z_grid).log()       #梯形法求积分得到 p(x)，最后取对数得到 log p(x)

print(f'重构项 E_q[log p(x|z)] ≈ {elbo_recon.item():.4f}')
print(f'KL(q||p)              = {elbo_kl.item():.4f}')
print(f'ELBO                  ≈ {elbo.item():.4f}')
print(f'log p(x) (数值积分)   ≈ {log_p_x.item():.4f}')

# 最后的验证不等式
print(f'验证: ELBO ({elbo.item():.4f})  <= log p(x) ({log_p_x.item():.4f})')  # ELBO 永远不会超过对数边缘似然，也就是 “下界” 的含义。

重构项 E_q[log p(x|z)] ≈ -1.4225
KL(q||p)              = 0.5000
ELBO                  ≈ -1.9225
log p(x) (数值积分)   ≈ -1.5155
验证: ELBO (-1.9225)  <= log p(x) (-1.5155)


要点：trapezoid 数值积分算出真实 log p(x)，你会发现 ELBO ≤ log p(x) 恒成立

log_p_xz = torch.distributions.Normal(zs, 1.0).log_prob(x)<br>
`Normal(z_grid, 1.0)`：<br>似然 $p(x|z)=\mathcal{N}(x;\ z,\, \sigma=1)$，均值就是网格上每一个 z，标准差固定为 1。<br>
`dist.log_prob(value)` <br>返回 **log (概率密度)**，也就是取完自然对数 $\ln$ 之后的结果。<br>
dist：一个概率分布对象，例如 `Normal(mu, sigma)` <br>
`prob` = probability，概率密度函数 p(x);`log` = 取自然对数 ln

elbo_recon = log_p_xz.mean()<br>
蒙特卡洛近似期望 $\mathbb{E}_{q(z|x)}[\log p(x|z)]$

elbo_kl = kl_gauss(mu_q, logvar_q, 0.0, 0.0)<br>
KL (q||p)；p (z) 是 N (0,1)

trapezoid = getattr(torch, 'trapezoid', torch.trapz)<br>
**梯形法数值积分函数**，作用：已知一堆离散点 (z, y)，估算曲线下面包围的面积<br>
$\int y(z)\,dz$ PyTorch **1.13 之后**：函数名叫 `torch.trapezoid(y, x)` <br>
- PyTorch **老版本**：名字叫 `torch.trapz(y, x)` <br>
`getattr(对象, "属性名字符串", 默认值)` <br>
规则：<br>
1. 如果 `torch` 里面存在名字叫 `'trapezoid'` 的函数 → 返回 `torch.trapezoid` <br>
2. 如果 torch 找不到 trapezoid（旧版本）→ 返回后备默认值 `torch.trapz` <br>
等价于 一个 if else 语句

elbo = elbo_recon - elbo_kl<br>
ELBO 证据下界定义：$\mathrm{ELBO}= \mathbb{E}_q[\log p(x|z)] - D_{\mathrm{KL}}(q\|p)$

# 四、小结测试

1. 为什么 p(x) = ∫p(x|z)p(z)dz 直接算不出来？(关键词：高维、intractable)<br>
因为我们这里的维度太高，会有非常多层积分要去算；且 p(x|z) 是高度非线性函数，非线性函数乘上先验p(z)，数学上找不到一个化简后的积分公式。没有办法写下一个简短表达式等于这个积分；且数值积分也死路——高维空间网格点随维度指数爆炸（维度灾难）；这才是 intractable 的完整含义：解析没闭式解，数值算不动。

2. 写出 ELBO 的两种等价形式，并说明为什么 log p(x) ≥ ELBO。<br>
形式一：原始定义（训练 VAE 的损失）<br>
$\mathrm{ELBO}(q)
=\mathbb{E}_{q(z|x)}\big[\log p(x|z)\big]
-D_{\mathrm{KL}}\big(q(z|x)\,\|\,p(z)\big)$  <br>
$\mathbb{E}_{q(z|x)}[\log p(x|z)]$：重构项 / 对数似然期望<br>
$D_{\mathrm{KL}}(q\|p)$：后验‑先验 KL 散度

形式二：带真实后验的 KL 形式（证据下界的由来式）<br>
$\mathrm{ELBO}(q)
=\log p(x)
-D_{\mathrm{KL}}\big(q(z|x)\,\|\,p(z|x)\big)%$  <br>
$p(z|x)$：真实后验，intractable、求不出来<br>
两个式子数学上完全相等，可以互相推导。<br>
log p(x)是想要最大化的目标，而 ELBO只是log p(x) 的一个可以计算的下界，形式二两边一移项就是 log p(x) = ELBO + KL ≥ ELBO，而 KL不为负，则不等式成立

3. KL 项为什么被称为"正则化"？如果只保留重构项、去掉 KL 项，模型会退化成什么？<br>
最小化 KL 等价于约束编码器输出的后验分布靠近预先设定的先验 N (0,1)，在损失函数上它是额外的惩罚项；<br>编码器趋向于让隐变量方差趋近于零，随机性消失，隐编码变成孤立点，随机生成新样本的能力彻底丢失。<br>
去掉 KL 只剩重构损失 → VAE 退化为普通自编码器。

4. 为什么对 z ~ q(z|x) 直接采样、再对 μ 求梯度会失败？梯度被什么挡住？<br>
因为直接采样，是对正态分布取一个常值，采样算子无梯度定义，路径断裂，得不到什么关于 μ 的表达式传回，那梯度就是会断的

5. 重参数化 z = μ + σ⊙ε 里 ε 承担了什么角色？为什么这样梯度就能回传？<br>
ε 承担了随机值的角色，且就是一个普通常数数值，它不需要梯度、也不会被优化；固定 ε 后，z 是 σ，μ  的显式可微函数，链式法则可用，则梯度可以回传给编码器。

6. 写出高斯 VAE 的 KL 项简化公式（先验 N(0,I) 时），并说清它惩罚的是后验的哪些性质。<br>
$D_{\mathrm{KL}}\big(q(z|x)\,\|\,p(z)\big)
=\frac12\sum_{i=1}^{d}\Big(\,\sigma_i^2+\mu_i^2-1-\log\sigma_i^2\,\Big)$ ;

**$\boldsymbol{\mu_i^2}$：惩罚后验均值离原点过远** <br>
   - 如果 $\mu_i$ 远离 0，该项变大，KL 损失上升。<br>
   - 作用：把编码器输出的均值往 0 方向拽。<br>
 **$\boldsymbol{\sigma_i^2}$：惩罚方差太大** <br>
   - $\sigma_i$ 很大，分布很宽 → 损失上升。<br>
   - 阻止后验散开到隐空间很远的区域。<br>
 **$\boldsymbol{-\log\sigma_i^2}$：惩罚方差太小** <br>
   - 当 $\sigma_i\to 0$，$\log\sigma_i^2\to-\infty$，于是 $-\log\sigma_i^2\to+\infty$，损失爆炸。<br>
   - 这一项**专门阻止方差坍缩到 0**，也就是防止编码器退化成确定性自编码器。<br>
 -1：常数偏移，不影响优化方向。

7. 训练完成后，想生成新样本，需要用到编码器吗？生成过程是什么？<br>
生成新样本时，不需要编码器。只使用解码器。从先验分布随机采样 p(z)=N(0,1) 隐向量,将采样得到的隐变量 z 送入解码器网络 Decoder,生成出来的新图像样本。

8. 编码器输出的为什么是 log σ² 而不是直接输出 σ？(提示：σ 的约束)<br>
标准差 σ 必须严格大于 0，而神经网络输出是无约束实数；直接输出 σ 有可能产生非法负值;输出 ，网络输出 log σ² 不受正数约束；后续用 exp() 还原方差，天然保证方差 > 0;log σ² 还有数值上的好处——方差跨数量级（0.01~100），对数刻度下梯度的尺度均匀，直接输出 σ 时小方差区域的梯度会被压扁，训练更慢。

9. ELBO 里的重构项是正的还是负的？log 似然最大能到多少？<br>
负的；0

10. 为什么是 KL(q||p) 而不是 KL(p||q)？两个方向有什么本质区别？(提示：zero-forcing / zero-avoiding)<br>
工程层次（最直接的理由）：<br>
KL(q(z|x)||p(z)) 里的两个分布都是已知、可算的——q 是我们自己控制的编码器，p 是先验 N(0,1)，闭式公式直接算（第 6 题那个公式）。反过来的 KL(p(z|x)||q) 需要 intractable 的真实后验 p(z|x)。所以不是"想选哪个方向"，是只有这一个方向可算。<br>
几何层次（行为差异）：<br>
KL(q||p) = zero-forcing / mode-seeking：q 必须避开 p 为零的区域，否则惩罚爆表 → q 倾向于收缩，只覆盖 p 的少数峰。缺点：方差欠估计（低估不确定性）。
KL(p||q) = zero-avoiding：q 必须覆盖 p 的全部支撑，否则在 p 有质量处 q 为 0 会惩罚爆表 → q 倾向于展宽，覆盖全域。缺点：均值模糊（低估 sharpness）。